# Apache Spark Architecture

**Apache Spark** is a unified, open-source analytics engine for large-scale data processing. It is the **core computational engine** behind Databricks.

## Key Characteristics
- **In-memory processing**: Caches intermediate data in RAM, avoiding repeated disk I/O (100× faster than MapReduce for certain workloads).
- **Unified engine**: Supports batch, streaming, ML, and graph workloads under one API.
- **Lazy evaluation**: Builds a logical plan before executing — enables optimization.
- **Fault tolerance**: Recomputes lost data using lineage (DAG) rather than replication.
- **Language support**: Python (PySpark), Scala, Java, R, SQL.

## Spark Cluster Architecture

```
┌──────────────────────────────────────────────────────────────┐
│                      SPARK APPLICATION                        │
│                                                              │
│  ┌─────────────────────────────────┐                         │
│  │         DRIVER NODE              │                         │
│  │  - SparkContext / SparkSession   │                         │
│  │  - Builds DAG (execution plan)   │                         │
│  │  - Schedules tasks               │                         │
│  │  - Tracks job progress           │                         │
│  └───────────────┬─────────────────┘                         │
│                  │  (communicates with)                       │
│  ┌───────────────▼─────────────────┐                         │
│  │       CLUSTER MANAGER            │                         │
│  │  (Databricks: managed by Azure   │                         │
│  │   Databricks Control Plane)      │                         │
│  └───────────────┬─────────────────┘                         │
│                  │  allocates resources                       │
│       ┌──────────┼──────────┐                                 │
│       ▼          ▼          ▼                                 │
│  ┌─────────┐ ┌─────────┐ ┌─────────┐                         │
│  │EXECUTOR │ │EXECUTOR │ │EXECUTOR │   (Worker Nodes)         │
│  │ Task    │ │ Task    │ │ Task    │                          │
│  │ Task    │ │ Task    │ │ Task    │                          │
│  │ Cache   │ │ Cache   │ │ Cache   │                          │
│  └─────────┘ └─────────┘ └─────────┘                         │
└──────────────────────────────────────────────────────────────┘
```

### Components

| Component | Role |
|-----------|------|
| **Driver** | The JVM process running the `main()` function of a Spark application. Hosts SparkSession, creates the DAG, schedules tasks on executors. |
| **SparkSession** | Entry point for all Spark functionality. Replaces older SparkContext, SQLContext, HiveContext. |
| **Cluster Manager** | Allocates resources across executors. Options: Spark Standalone, YARN, Kubernetes, Mesos. In Databricks, managed automatically. |
| **Executor** | JVM process on a worker node. Executes tasks, stores RDD/DataFrame partitions in memory or disk. |
| **Task** | The smallest unit of work in Spark. One task processes one **partition** of data. |
| **Slot** | A thread within an executor that runs one task at a time. `Total slots = executors × cores per executor`. |

## Execution Model: Jobs, Stages, and Tasks

```
Application
  └── Job (one per Action)
        └── Stage (split by shuffle boundaries)
              └── Task (one per partition)
```

| Unit | Triggered By | Description |
|------|-------------|-------------|
| **Job** | An **Action** (e.g., `.count()`, `.collect()`, `.write`) | Complete computation to produce a result |
| **Stage** | **Shuffle boundary** (wide transformation) | A set of tasks that can run without data exchange between nodes |
| **Task** | Each **partition** of data | The unit that actually runs on an executor core |

### Narrow vs. Wide Transformations

| Type | Description | Shuffle? | Examples |
|------|-------------|----------|----------|
| **Narrow** | Each partition of the parent produces one partition of the child. No data movement across nodes. | ❌ No | `filter()`, `map()`, `select()`, `withColumn()` |
| **Wide** | Multiple parent partitions contribute to one child partition. Requires a **shuffle** (network I/O). | ✅ Yes | `groupBy()`, `join()`, `distinct()`, `orderBy()`, `repartition()` |

> ⚠️ **Shuffles are expensive** — they involve serializing data, writing to disk, and transferring over the network. Minimize unnecessary wide transformations.

## Directed Acyclic Graph (DAG) & Lazy Evaluation

Spark uses **lazy evaluation** — transformations are not executed immediately. Instead, Spark builds a **DAG** (logical plan) and only executes the full plan when an **action** is called.

### Transformations vs. Actions

| Category | Description | Triggers Execution? | Examples |
|----------|-------------|---------------------|----------|
| **Transformation** | Returns a new DataFrame/RDD. Builds the DAG. | ❌ No | `select()`, `filter()`, `join()`, `groupBy()`, `withColumn()` |
| **Action** | Triggers execution of the DAG, returns a result to the driver or writes to storage. | ✅ Yes | `show()`, `count()`, `collect()`, `write.save()`, `take(n)` |

### Why Lazy Evaluation?
- Allows the **Catalyst Optimizer** to optimize the entire query plan before execution.
- Avoids unnecessary computation — only computes what is needed.
- Enables **predicate pushdown**, **column pruning**, and other optimizations automatically.

## Catalyst Optimizer & Tungsten Engine

These are Spark's two internal optimization components that make DataFrames and Spark SQL highly performant.

### Catalyst Optimizer (Query Planning)

```
User Code (DataFrame / SQL)
        │
        ▼
  Unresolved Logical Plan
        │  (resolve column names, types)
        ▼
  Resolved Logical Plan
        │  (apply optimization rules)
        ▼
  Optimized Logical Plan
        │  (generate physical plans)
        ▼
  Physical Plans (multiple candidates)
        │  (cost-based selection)
        ▼
  Selected Physical Plan → Code Generation → Execution
```

| Optimization Technique | Description |
|------------------------|-------------|
| **Predicate pushdown** | Filters applied as early as possible (even pushed to the data source layer, e.g., Parquet/Delta) |
| **Column pruning** | Only reads the columns actually needed |
| **Join reordering** | Reorders joins to minimize intermediate data size |
| **Constant folding** | Evaluates constant expressions at compile time |

### Tungsten Engine (Physical Execution)
- **Whole-stage code generation**: Compiles multiple operations into single fused JVM bytecode, eliminating virtual function calls.
- **Off-heap memory management**: Manages memory directly, bypassing JVM garbage collection overhead.
- **Cache-aware computation**: Organizes data in memory to improve CPU cache hit rates.

## Spark Data Abstractions

Spark has evolved through three data abstractions:

### 1. RDD (Resilient Distributed Dataset) — Spark 1.x
- Low-level API. Distributed collection of objects.
- **Resilient**: Recomputes lost partitions using lineage.
- **Distributed**: Partitioned across cluster nodes.
- No schema; developer manages types.
- **When to use**: Unstructured data, fine-grained control over physical execution, UDFs with complex Python objects.

### 2. DataFrame — Spark 2.x (preferred)
- Distributed collection of **rows with a named, typed schema**.
- Optimized via Catalyst + Tungsten automatically.
- SQL-like API. Equivalent to a database table.
- **Default choice** for all structured and semi-structured workloads.

### 3. Dataset — Spark 2.x (Scala/Java only)
- Type-safe version of DataFrame.
- Not available in PySpark (Python is dynamically typed).

| Feature | RDD | DataFrame | Dataset |
|---------|-----|-----------|----------|
| **Type safety** | ✅ (compile-time) | ❌ (runtime) | ✅ (compile-time) |
| **Schema** | None | Yes | Yes |
| **Optimization** | None | Catalyst | Catalyst |
| **Language** | All | All | Scala/Java only |
| **Performance** | Lowest | High | High |
| **Recommended** | Rarely | ✅ Yes | Scala only |

## Partitioning

**Partitions** are the fundamental unit of parallelism in Spark. Each partition is processed by one task on one executor core.

### Key Rules
- Default partition size: **128 MB** (configurable via `spark.sql.files.maxPartitionBytes`).
- Default shuffle partitions: **200** (configurable via `spark.sql.shuffle.partitions`).
- In Databricks with **Adaptive Query Execution (AQE)**, shuffle partitions are automatically optimized.

| Scenario | Effect | Fix |
|----------|--------|-----|
| **Too few partitions** | Under-utilizes cluster, tasks take too long | `repartition(n)` to increase |
| **Too many partitions** | Task scheduling overhead dominates | `coalesce(n)` to reduce (no shuffle) |
| **Data skew** | One partition much larger than others → one slow task | Salt key, AQE skew handling |

### `repartition()` vs. `coalesce()`

| Method | Shuffle? | Use When |
|--------|----------|----------|
| `repartition(n)` | ✅ Yes (full shuffle) | Increasing partitions, evening out skew |
| `coalesce(n)` | ❌ No (narrow) | Reducing partitions efficiently before write |

### Delta Lake Optimization (Databricks)
- **OPTIMIZE**: Compacts small files into larger ones (right-sizes partitions on disk).
- **Z-ORDER**: Co-locates related data within files for faster filtering on specified columns.
- **Auto Optimize**: Automatically runs optimization in Databricks (auto-compact + optimized writes).

## Spark Memory Management

Each **executor JVM** divides its memory into regions:

```
┌───────────────────────────────────────────────┐
│              Executor JVM Memory               │
│                                               │
│  ┌─────────────────────────────────────────┐  │
│  │         Unified Memory Region (60%)      │  │
│  │                                         │  │
│  │  ┌──────────────┐  ┌──────────────────┐ │  │
│  │  │  Storage     │  │   Execution      │ │  │
│  │  │  Memory      │◄─►   Memory         │ │  │
│  │  │ (cache, RDDs)│  │ (shuffle, sort,  │ │  │
│  │  │              │  │  aggregations)   │ │  │
│  │  └──────────────┘  └──────────────────┘ │  │
│  └─────────────────────────────────────────┘  │
│  ┌─────────────────────────────────────────┐  │
│  │       Reserved / User Memory (40%)       │  │
│  │  (Python UDFs, driver code, JVM overhead)│  │
│  └─────────────────────────────────────────┘  │
└───────────────────────────────────────────────┘
```

### Storage vs. Execution Memory
- **Storage memory**: Used for caching DataFrames/RDDs (`df.cache()`, `df.persist()`).
- **Execution memory**: Used for shuffle data, sorting, aggregations.
- In **Unified Memory Model** (Spark 1.6+), these two pools share a unified region and borrow from each other dynamically.

### Caching
| Method | Storage Level | Use When |
|--------|---------------|----------|
| `df.cache()` | MEMORY_AND_DISK | Shorthand; reused DataFrames |
| `df.persist(StorageLevel.MEMORY_ONLY)` | RAM only | Fits in memory |
| `df.persist(StorageLevel.DISK_ONLY)` | Disk only | Does not fit in memory |
| `df.unpersist()` | — | Free memory when no longer needed |

> ⚠️ In Databricks, **Delta caching** (disk-based SSD cache on worker nodes) automatically caches Parquet/Delta data read from cloud storage — separate from Spark's in-memory cache.